# Gemma 4 Trust & Safety Context Pruning Runner

Use this notebook inside Kaggle for Gemma 4 Trust & Safety research experiments.
It runs the ACPA pipeline, creates benchmark artifacts, and writes JSONL output.

Expected Kaggle setup:

1. Attach the Agentic Eval dataset to this notebook.
2. Add your Gemma/Gemini API key through Kaggle Secrets using label `GEMINI_API_KEY`.
3. Enable Kaggle notebook internet if you want the first setup cell to clone the GitHub repo automatically.
4. Run cells from top to bottom.

Do not paste API keys into public notebook cells or commit them to Git.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/joyjeni/context-pruning.git'
REPO_BRANCH = 'cursor/gemma-acpa-trust-safety-44f2'
CLONE_DIR = Path('/kaggle/working/context-pruning') if Path('/kaggle/working').exists() else Path.cwd()

possible_roots = [
    Path.cwd(),
    CLONE_DIR,
    Path('/kaggle/input/context-pruning'),
]

REPO_ROOT = next((path for path in possible_roots if (path / 'src/acpa_gemma').exists()), None)

if REPO_ROOT is None:
    print('Repository source not found. Cloning into', CLONE_DIR)
    if CLONE_DIR.exists() and CLONE_DIR.name == 'context-pruning':
        print('Removing stale clone directory:', CLONE_DIR)
        shutil.rmtree(CLONE_DIR)
    try:
        subprocess.run(
            [
                'git',
                'clone',
                '--depth',
                '1',
                '--branch',
                REPO_BRANCH,
                REPO_URL,
                str(CLONE_DIR),
            ],
            check=True,
        )
    except Exception as exc:
        raise RuntimeError(
            'Could not clone the repository. In Kaggle, turn Internet ON in '
            'Notebook settings, then rerun this cell. If Internet is disabled, '
            'upload the repository as a Kaggle dataset and set REPO_ROOT to that path.'
        ) from exc
    REPO_ROOT = CLONE_DIR

SRC_ROOT = REPO_ROOT / 'src'
if not (SRC_ROOT / 'acpa_gemma').exists():
    raise RuntimeError(f'acpa_gemma source package not found under {SRC_ROOT}')

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('REPO_ROOT =', REPO_ROOT)
print('SRC_ROOT exists =', SRC_ROOT.exists())
print('acpa_gemma exists =', (SRC_ROOT / 'acpa_gemma').exists())
print('Kaggle input exists =', Path('/kaggle/input').exists())


## Dependency install

Run this cell as normal Python. Do **not** type `pip install ...` without a leading `!` in a Kaggle code cell; that causes `SyntaxError`. The cell below uses `python -m pip` through `subprocess`, so it works as valid Python. Internet must be enabled in the notebook settings if packages are missing.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'google.genai': 'google-genai',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All required packages are already installed.')


## Locate Agentic Eval dataset

If auto-detection picks the wrong path, set `DATASET_DIR` manually to the attached dataset directory.


In [ ]:
input_root = Path('/kaggle/input')
if input_root.exists():
    candidates = sorted([p for p in input_root.iterdir() if p.is_dir()])
    print('Available /kaggle/input datasets:')
    for candidate in candidates:
        print(' -', candidate)
    DATASET_DIR = next((p for p in candidates if 'agent' in p.name.lower() and 'eval' in p.name.lower()), candidates[0] if candidates else input_root)
else:
    DATASET_DIR = Path('demo-missing-kaggle-input')

print('DATASET_DIR =', DATASET_DIR)


## Add and verify your Google API key using Kaggle Secrets

Write the secret in the Kaggle UI, not in this notebook:

1. Open the notebook right sidebar.
2. Click **Add-ons**.
3. Click **Secrets**.
4. Add a secret with label `GEMINI_API_KEY` and value equal to your Google AI Studio / Gemini API key.
5. Turn on notebook access for that secret.

The code cell below is where you verify the secret and write `/kaggle/working/configs/secrets.toml`, which is what the pipeline reads. It prints only `True`/`False`, never the key itself.


In [ ]:
config_dir = Path('/kaggle/working/configs') if Path('/kaggle/working').exists() else REPO_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

# This label must exactly match the label you added in Kaggle Add-ons -> Secrets.
secret_label = 'GEMINI_API_KEY'
api_key = ''
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret(secret_label)
except Exception as exc:
    print(f'Could not load Kaggle Secret {secret_label!r}: {exc}')

print('API key loaded from Kaggle Secrets =', bool(api_key))

app_toml = f'''
[gemma]
model = "gemma-4-26b-a4b-it"
temperature = 0.2
top_p = 0.9
max_output_tokens = 2048

[data]
input_dir = "{DATASET_DIR}"
sample_size = 0

[pruning]
alpha = 1.5
beta = 1.0
gamma = 0.5
delta = 10.0
prune_ratio = 0.45
cache_threshold = 2
priority_boost = 1.5

[output]
path = "/kaggle/working/results.jsonl"
'''.strip() + "\n"

secrets_toml = f'''
[gemma]
api_key = "{api_key}"
'''.strip() + "\n"

(config_dir / 'app.toml').write_text(app_toml, encoding='utf-8')
(config_dir / 'secrets.toml').write_text(secrets_toml, encoding='utf-8')

print('Wrote', config_dir / 'app.toml')
print('Wrote', config_dir / 'secrets.toml')
print('API key present =', bool(api_key))


## Dry-run pipeline test, no API key required

This verifies dataset loading, context construction, ACPA pruning, and JSONL output.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/dry_run.jsonl',
    '--sample-size', '3',
    '--dry-run',
])

print(Path('/kaggle/working/dry_run.jsonl').read_text(encoding='utf-8')[:2000])


## Benchmark ACPA vs baseline algorithms, no API key required

Creates per-record CSV, aggregate CSV, and a Markdown report.


In [ ]:
from acpa_gemma.benchmark import main as benchmark_main

benchmark_main([
    '--input', str(DATASET_DIR),
    '--sample-size', '100',
    '--details-output', '/kaggle/working/benchmark_details.csv',
    '--summary-output', '/kaggle/working/benchmark_summary.csv',
    '--report-output', '/kaggle/working/benchmark_report.md',
])

print(Path('/kaggle/working/benchmark_report.md').read_text(encoding='utf-8'))


## Real Gemma 4 run

Run this after `API key present = True`. Start with a small sample, then remove `--sample-size` for the full run.


In [ ]:
from acpa_gemma.cli import main as pipeline_main

if not api_key:
    raise RuntimeError('Add GEMINI_API_KEY/GEMMA_API_KEY in Kaggle Secrets or paste api_key in the config cell first.')

pipeline_main([
    '--config', str(config_dir / 'app.toml'),
    '--secrets', str(config_dir / 'secrets.toml'),
    '--input', str(DATASET_DIR),
    '--output', '/kaggle/working/results_sample.jsonl',
    '--sample-size', '10',
])

print(Path('/kaggle/working/results_sample.jsonl').read_text(encoding='utf-8')[:3000])


## Full results run

Uncomment this cell when the sample run looks good.


In [ ]:
# pipeline_main([
#     '--config', str(config_dir / 'app.toml'),
#     '--secrets', str(config_dir / 'secrets.toml'),
#     '--input', str(DATASET_DIR),
#     '--output', '/kaggle/working/results.jsonl',
# ])


## Output files

Attach these files to your research report, hackathon demo, or publication appendix as needed:

- `/kaggle/working/results.jsonl`
- `/kaggle/working/results_sample.jsonl`
- `/kaggle/working/benchmark_report.md`
- `/kaggle/working/benchmark_summary.csv`
- `/kaggle/working/benchmark_details.csv`
